<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [ ]:
import torch.nn as nn
proj = nn.Conv1d(
            in_channels=3,
            out_channels=512*3, # if shared embedding, then d_model is the output dimension
            kernel_size=100,
            stride=100,
            padding=0,
            groups=3 # added to handle multiple channels / keep them separate
        )

import torch

W_P = nn.ModuleList()
for _ in range(3): W_P.append(nn.Linear(100, 512))

x = torch.randn(10,3,1000).shape

In [ ]:
l = nn.Linear(100,512)

In [ ]:
l.weight.shape, l.bias.shape

(torch.Size([512, 100]), torch.Size([512]))

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(W_P), count_parameters(proj)

(155136, 155136)

In [0]:
#| echo: false
#| output: asis
show_doc(InceptionTokenizer)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/tokenizers.py#L137){target="_blank" style="float:right; font-size:smaller"}

### InceptionTokenizer

```python
def InceptionTokenizer(
    c_in, # the number of input channels
    patch_size, # the length of the patches (either stft or interval length)
    d_model, # the dimension of the initial linear layers for inputting patches into transformer
    patch_stride:NoneType=None, # the stride of the patches
    shared_embedding:bool=True, **tokenizer_kwargs
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(LinearTokenizer)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/tokenizers.py#L91){target="_blank" style="float:right; font-size:smaller"}

### LinearTokenizer

```python
def LinearTokenizer(
    c_in, # the number of input channels
    patch_size, # the length of the patches (either stft or interval length)
    d_model, # the dimension of the initial linear layers for inputting patches into transformer
    shared_embedding:bool=False, # indicator of whether to project each channel individually or together
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(TS_Tokenizer_Complex)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/tokenizers.py#L58){target="_blank" style="float:right; font-size:smaller"}

### TS_Tokenizer_Complex

```python
def TS_Tokenizer_Complex(
    c_in, patch_size, d_model, constant_pad_value:float=0.0
):
```

*Time series 2D convolutional Embedding*

In [0]:
#| echo: false
#| output: asis
show_doc(TS_Tokenizer)

/gpfs/home/dk5565/.conda/envs/physiojepa/lib/python3.10/site-packages/fastcore/docscrape.py:259: UserWarning: potentially wrong underline length... 
Tokenizer class based on a Conv1D 
--- in 
Tokenizer class based on a Conv1D
---...
  else: warn(msg)


---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/tokenizers.py#L13){target="_blank" style="float:right; font-size:smaller"}

### TS_Tokenizer

```python
def TS_Tokenizer(
    c_in, patch_size, d_model, patch_stride:NoneType=None, shared_embedding:bool=True
):
```

*Tokenizer class based on a Conv1D*
---
    c_in (int): Number of input channels
    patch_size (int): Size of each patch/kernel
    d_model (int): Output embedding dimension

In [0]:
#| echo: false
#| output: asis
show_doc(MultiScaleTokenizer)

/gpfs/home/dk5565/.conda/envs/physiojepa/lib/python3.10/site-packages/fastcore/docscrape.py:259: UserWarning: potentially wrong underline length... 
Output shape matches other tokenizers: [bs, num_patches, n_vars, d_model] 
--- in 
Hierarchical multi-scale tokenizer that combines fine-grained local encoding
with coarse patch tokenization....
  else: warn(msg)


---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/tokenizers.py#L186){target="_blank" style="float:right; font-size:smaller"}

### MultiScaleTokenizer

```python
def MultiScaleTokenizer(
    c_in, patch_size, d_model, patch_stride:NoneType=None, shared_embedding:bool=True, fine_patch_size:int=25,
    fine_d_model:int=128, fine_n_heads:int=4, fine_d_ff:int=512, fine_layers:int=1, fine_dropout:float=0.0
):
```

*Hierarchical multi-scale tokenizer that combines fine-grained local encoding*
with coarse patch tokenization.

Fine path: splits each coarse patch interval into `group_size` sub-patches,
processes them with a lightweight local transformer, and pools each group
into a single summary vector.

Coarse path: standard TS_Tokenizer (Conv1d) over the full signal.

Fusion: additive combination of projected fine summaries and coarse tokens.

Output shape matches other tokenizers: [bs, num_patches, n_vars, d_model]
---
    c_in (int): Number of input channels
    patch_size (int): Coarse patch size (e.g. 125 for 1s at 125Hz)
    d_model (int): Output embedding dimension (coarse/global)
    patch_stride (int): Coarse patch stride (default=patch_size)
    shared_embedding (bool): Whether channels share embedding weights
    fine_patch_size (int): Fine sub-patch size in samples (default=25)
    fine_d_model (int): Fine encoder hidden dimension (default=128)
    fine_n_heads (int): Attention heads in local encoder (default=4)
    fine_d_ff (int): Feed-forward width in local encoder (default=512)
    fine_layers (int): Number of local transformer layers (default=1)
    fine_dropout (float): Dropout in local encoder (default=0.0)

In [0]:
#| echo: false
#| output: asis
show_doc(PatchEncoder)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/tokenizers.py#L382){target="_blank" style="float:right; font-size:smaller"}

### PatchEncoder

```python
def PatchEncoder(
    c_in, # the number of input channels
    patch_len, # the length of the patches (either stft or interval length)
    d_model, # the dimension of the initial linear layers for inputting patches into transformer
    shared_embedding, # indicator of whether to project each channel individually or together
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool